#### calculate accuracy and iaa for metaphor representations

In [66]:
# imports
import pandas as pd
import numpy as np
import pandas as pd
import krippendorff

from src.utils import load_env, load_json, get_logger

# variables
DATASET = "tweets_immigration"
if DATASET == "tweets_immigration":
    DATASET_OUT_NAME = "tweets_immigration_labeled"
else:
    DATASET_OUT_NAME = DATASET
env_vars = load_env()
logger = get_logger("met-rep-eval")

In [67]:
# load data - both annotators and adjudications
completed_anns_path = f"{env_vars['RESULTS_DIR']}/annotation/lm_representation/completed_anns"
base_filename = f"{DATASET_OUT_NAME}_100_to_annotate"

ann0_df = pd.read_csv(f"{completed_anns_path}/{base_filename} - ann0.csv", dtype=str)
ann1_df = pd.read_csv(f"{completed_anns_path}/{base_filename} - ann1.csv", dtype=str)

ann0_df = ann0_df.apply(lambda col: col.str.upper() if col.dtype == "object" else col).drop(columns=['notes'])
ann1_df = ann1_df.apply(lambda col: col.str.upper() if col.dtype == "object" else col).drop(columns=['notes'])

In [68]:
ann0_df.columns

Index(['metaphor_id', 'sentence', 'target_word', 'noun_group', 'target_group',
       'source_word', 'source_schema', 'frame_implications', 'correct_ng?',
       'correct_tg?', 'correct_ss?', 'plausible_implications?',
       'metaphor_implications?'],
      dtype='str')

In [69]:
def get_val_score(row, col):
    val = row[col]
    if pd.isna(val):
        if "correct_tg" in col:
            return 1
        print("Warning! Missing annotation")
        return 0
    
    elif isinstance(val, (bool, np.bool_)):
        if bool(val):
            return 1
        else:
            return 0
    
    s = str(val).strip()
    if s.upper() == "TRUE":
        return 1
    else: 
        return 0

def calculate_rep_score(r):
    return int(get_val_score(r, "correct_ng?") + get_val_score(r, "correct_tg?") 
        + get_val_score(r, "correct_ss?") + get_val_score(r, "plausible_implications?") + get_val_score(r, "metaphor_implications?"))

def calculate_text_score(r):
    return int(get_val_score(r, "plausible_implications?") + get_val_score(r, "metaphor_implications?"))

def calculate_disc_score(r):
    return int(get_val_score(r, "correct_ng?") + get_val_score(r, "correct_tg?") + get_val_score(r, "correct_ss?"))


ann0_df["disc_score"] = ann0_df.apply(calculate_disc_score, axis=1)
ann1_df["disc_score"] = ann1_df.apply(calculate_disc_score, axis=1)

ann0_df["text_score"] = ann0_df.apply(calculate_text_score, axis=1)
ann1_df["text_score"] = ann1_df.apply(calculate_text_score, axis=1)

ann0_df["rep_score"] = ann0_df.apply(calculate_rep_score, axis=1)
ann1_df["rep_score"] = ann1_df.apply(calculate_rep_score, axis=1)

In [70]:
ann_df = ann0_df.merge(ann1_df, on=["metaphor_id", "sentence", "target_word", "target_group", "noun_group", "source_word", "source_schema", "frame_implications"])
ann_df = ann_df.dropna(subset=["correct_ng?_y", "correct_ng?_x"])
print(len(ann_df))
ann_df.head()

100


,metaphor_id,sentence,target_word,noun_group,target_group,source_word,source_schema,frame_implications,correct_ng?_x,correct_tg?_x,...,text_score_x,rep_score_x,correct_ng?_y,correct_tg?_y,correct_ss?_y,plausible_implications?_y,metaphor_implications?_y,disc_score_y,text_score_y,rep_score_y
0,1063307199479468033_0_14_12,So says there is a perception of Ice being li...,that,organization,Ku Klux Klan,prohibiting,other,Illegal immigrants are being denied voting rig...,TRUE,TRUE,...,0,3,FALSE,FALSE,TRUE,FALSE,Not Metaphorical,1,0,1
1,1116197982335066112_2_6_11,Isn’t marrying your brother to skirt our immig...,crime,thing,Anti-Immigration Sentiment,skirt,spatial motion,The immigration system is being circumvented t...,TRUE,FALSE,...,2,4,TRUE,FALSE,TRUE,TRUE,TRUE,2,2,4
2,1034405682123034624_2_12_15,Wait patiently for him to spring the trap he h...,part,thing,Political Rhetoric and Partisan Debate,drain,force,The DOJ's involvement in the swamp is a harmfu...,TRUE,FALSE,...,2,4,TRUE,FALSE,TRUE,TRUE,TRUE,2,2,4
3,1098619955023568896_2_3_0,That doesn't stop the illegals.',that,thing,Anti-Immigration Sentiment,stop,other,There is an ongoing influx of undocumented imm...,TRUE,FALSE,...,2,3,TRUE,FALSE,FALSE,TRUE,Not Metaphorical,1,1,2
4,1020767613666955265_1_15_17,"Just look at his accomplishments, $20 Trillion...",economy,thing,Anti-Immigration Sentiment,running,force,Illegal immigrants are overwhelming the econom...,TRUE,FALSE,...,2,4,TRUE,FALSE,FALSE,TRUE,TRUE,1,2,3


In [71]:
"""
Compute Krippendorff's alpha, Gwet's AC1, and percent agreement between
annotator_x and annotator_y columns.

Requires: pip install krippendorff irrCAC pandas

Assumes your dataframe has paired columns like:
    correct_ng?_x / correct_ng?_y
    correct_tg?_x / correct_tg?_y
    ...
and computes nominal-level Krippendorff's alpha, Gwet's AC1, and raw percent
agreement for each pair.
"""


def percent_agreement(df, col_x, col_y):
    """Compute raw percent agreement between two annotator columns (ignoring rows where either is missing)."""
    x = df[col_x].apply(normalize_value, col=col_x)
    y = df[col_y].apply(normalize_value, col=col_y)

    valid = x.notna() & y.notna()
    n_valid = valid.sum()
    if n_valid == 0:
        return np.nan, 0

    agreement = (x[valid] == y[valid]).mean()
    return agreement, n_valid


def alpha_for_pair(df, col_x, col_y, level_of_measurement="nominal"):
    """Compute Krippendorff's alpha for a single pair of annotator columns."""
    x = df[col_x].apply(normalize_value, col=col_x)
    y = df[col_y].apply(normalize_value, col=col_y)

    # Map categorical values to numeric codes (krippendorff wants hashable/numeric,
    # NaN is preserved as missing so it's excluded automatically)
    all_vals = pd.unique(pd.concat([x, y]).dropna())
    val_to_code = {v: i for i, v in enumerate(all_vals)}

    def to_code(v):
        return np.nan if pd.isna(v) else val_to_code[v]

    reliability_data = [
        [to_code(v) for v in x.tolist()],
        [to_code(v) for v in y.tolist()],
    ]

    return krippendorff.alpha(
        reliability_data=reliability_data,
        level_of_measurement=level_of_measurement,
    )


def gwet_ac1_for_pair(df, col_x, col_y):
    """Compute Gwet's AC1 for a single pair of annotator columns (nominal, 2 raters)."""
    x = df[col_x].apply(normalize_value, col=col_x)
    y = df[col_y].apply(normalize_value, col=col_y)

    valid = x.notna() & y.notna()
    x, y = x[valid], y[valid]
    n = len(x)

    if n == 0:
        return np.nan

    categories = pd.unique(pd.concat([x, y]))
    q = len(categories)

    if q < 2:
        return np.nan  # no variability at all -> AC1 undefined

    # Observed percent agreement
    p_a = (x.values == y.values).mean()

    # Mean proportion of each category across both raters combined
    pi_k = pd.concat([x, y]).value_counts(normalize=True)

    # Chance agreement term (Gwet's formula)
    p_e = sum(pi_k[k] * (1 - pi_k[k]) for k in categories) / (q - 1)

    if p_e == 1:
        return 1.0 if p_a == 1 else np.nan

    return (p_a - p_e) / (1 - p_e)



def compute_all_alphas(df, skip_prefixes=("notes",), level_of_measurement="nominal"):
    """
    Find all _x/_y column pairs (excluding ones starting with skip_prefixes,
    e.g. free-text 'notes' columns which aren't suited to alpha) and compute
    Krippendorff's alpha, Gwet's AC1, and percent agreement for each.
    """
    results = {}
    x_cols = [c for c in df.columns if c.endswith("_x")]

    for col_x in x_cols:
        base = col_x[:-2]  # strip trailing "_x"
        col_y = base + "_y"

        if col_y not in df.columns:
            continue
        if any(base.startswith(p) for p in skip_prefixes):
            continue

        alpha = alpha_for_pair(df, col_x, col_y, level_of_measurement)
        ac1 = gwet_ac1_for_pair(df, col_x, col_y)
        pct_agree, n_valid = percent_agreement(df, col_x, col_y)

        results[base] = {
            "alpha": alpha.round(3),
            "ac1": ac1.round(3),
            "pct_agreement": pct_agree.round(3),
            "n": n_valid.round(3),
        }

    return results


def normalize_value(val, col):

    """Normalize messy True/TRUE/False/FALSE/NaN values into consistent strings."""
    if pd.isna(val):
        if "correct_tg" in col:
            return "True"
        print(f"Error! Null value in col {col}")
        raise ValueError
    
    elif isinstance(val, (bool, np.bool_)):
        return str(bool(val))
    
    s = str(val).strip()
    if s.upper() == "TRUE":
        return "True"
    if s.upper() == "FALSE":
        return "False"
    if s.upper() == "UNSURE":
        return "False"
    if s.upper() == "NOT METAPHORICAL":
        return "Not Metaphorical"
    if s in ["0", "1", "2", "3", "4", "5"]:
        return s
    else:
        print(f"'{s}' not a valid annotation")
        raise ValueError



df = ann_df
results = compute_all_alphas(df)

In [72]:
out_names = {
    "correct_ng?": "Noun Group Score",
    "correct_tg?": "Target Group Score",
    "correct_ss?": "Source Image Schema Group Score", 
    "plausible_implications?": "Interpretation Plausibility Score",
    "metaphor_implications?": "Interpretation Relevancy Score", 
    "disc_score": "Discrete Properties Score",
    "text_score": "Textual Interpretation Score",
    "rep_score": "Full Interpretation Score"
}


print(f"{'Annotation':35s} {'K-Alpha':>8s} {"Gwet's AC1":>15s} {'% Agreement':>14s} {'N':>6s}")
print("-" * 85)
for col, r in results.items():
    print(
        f"{out_names[col]:35s} {r['alpha']:8.3f} {r['ac1']:14.3f} "
        f"{r['pct_agreement']*100:13.0f}% {r['n']:7d}"
    )

Annotation                           K-Alpha      Gwet's AC1    % Agreement      N
-------------------------------------------------------------------------------------
Noun Group Score                       0.162          0.912            92%     100
Target Group Score                     0.532          0.550            77%     100
Source Image Schema Group Score       -0.015          0.286            58%     100
Interpretation Plausibility Score      0.083          0.638            74%     100
Interpretation Relevancy Score         0.292          0.667            73%     100
Discrete Properties Score              0.269          0.419            54%     100
Textual Interpretation Score           0.093          0.420            56%     100
Full Interpretation Score              0.141          0.289            41%     100


##### AC1 Interpretation
0.81 – 1.00: Almost perfect agreement<br>
0.61 – 0.80: Substantial agreement<br>
0.41 – 0.60: Moderate agreement<br>
0.21 – 0.40: Fair agreement<br>
0.00 – 0.20

##### now accuracy!

In [73]:
adj_df = pd.read_csv(f"{completed_anns_path}/{base_filename} - adjudications.csv", dtype=str)
adj_df["rep_score"] = adj_df.apply(calculate_rep_score, axis=1)
adj_df["text_score"] = adj_df.apply(calculate_text_score, axis=1)
adj_df["disc_score"] = adj_df.apply(calculate_disc_score, axis=1)

In [74]:
# print mean scores
print(f"rep score: {adj_df["rep_score"].mean().round(3)}/5")
print(f"text score: {adj_df["text_score"].mean().round(3)}/2")
print(f"disc score: {adj_df["disc_score"].mean().round(3)}/3")


rep score: 3.76/5
text score: 1.51/2
disc score: 2.25/3


In [75]:
for col in adj_df.columns:
    if col.endswith('?'):
        col_scores = [normalize_value(v, col) for v in adj_df[col].tolist()]
        av_score = float(col_scores.count('True')) / float(len(col_scores))
        print(f"{col}: {av_score}/1")


correct_ng?: 0.95/1
correct_tg?: 0.5/1
correct_ss?: 0.8/1
plausible_implications?: 0.75/1
metaphor_implications?: 0.76/1
